# CRNN + CTC - Digit Sequence Reader

Non-autoregressive parallel decoder that generalises to sequence lengths
far beyond the training distribution. The 2D CNN guarantees the model
never sees the *whole* image at once, and the 1D Dilated Residual CNN
(receptive field ~60 steps) prevents it from learning a length prior.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repository and Install Requirements

In [ ]:
REPO_URL = "https://github.com/AmineAitLaamim/digit-sequence-reader.git"
!git clone {REPO_URL}
%cd digit-sequence-reader
!pip install -r requirements.txt -q
!pip install albumentations opencv-python-headless -q

## 3. Verify GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device       :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## 4. (Optional) Shape Sanity-Check

Before kicking off a multi-hour training run, run the built-in
`__main__` block in `src/ctc/model.py`. It instantiates the CRNN_CTC
model and asserts that an input of shape `[2, 1, 64, 160]` produces
logits of shape `[2, 20, 11]` and a finite scalar CTC loss.

In [ ]:
!python -m src.ctc.model

## 5. Train the CTC model

Trains for `epochs` (default 30) with a curriculum that grows the
max sequence length from 7 to 12 digits. Validation CER and sequence
accuracy are logged to `metrics/ctc_metrics.csv`. The best checkpoint
is saved to `checkpoints_ctc/best_ctc.pt` (separate folder so it does
not overwrite the seq2seq model).

In [ ]:
DRIVE_PATH = "/content/drive/MyDrive/digit-sequence-reader"
BEST_CKPT  = f"{DRIVE_PATH}/checkpoints_ctc/best_ctc.pt"

import os
resume_flag = f'--resume "{BEST_CKPT}"' if os.path.exists(BEST_CKPT) else ''

!python -m src.ctc.train \
    --drive_path "{DRIVE_PATH}" \
    --epochs 30 \
    --batch_size 64 \
    --lr 1e-3 \
    {resume_flag}

## 6. Inspect the Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"{DRIVE_PATH}/metrics/ctc_metrics.csv")
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(df['epoch'], df['train_loss'], label='train')
axes[0].plot(df['epoch'], df['val_loss'],   label='val')
axes[0].set_title('CTC Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(df['epoch'], df['train_seq_acc'], label='train')
axes[1].plot(df['epoch'], df['val_seq_acc'],   label='val')
axes[1].set_title('Sequence Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(df['epoch'], df['train_char_acc'], label='train')
axes[2].plot(df['epoch'], df['val_char_acc'],   label='val')
axes[2].set_title('Character Accuracy (1 - CER)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.show()

## 7. Generate Sample Images (for inference demos)

Generates 20 PNGs of CTC-style sequences into `./samples/`. Each image's
width is guaranteed to be at least `num_digits * 16` so the 8x-CNN
downsampled time axis is at least `2 * num_digits`.

In [ ]:
!python -m src.ctc.generate_samples

## 8. Length-Extrapolation Sanity Check (the key selling point of CTC)

Run the trained model on a *very long* sequence it has never seen during
training. If length generalisation works, the CTC model should still
decode it correctly - the autoregressive seq2seq would not.

In [ ]:
import torch
import sys, os
sys.path.insert(0, '.')

from src.ctc.model      import CRNN_CTC, greedy_decode
from src.ctc.dataset    import build_multidigit_bank, get_digit_aug_pipeline, make_sequence
from src.ctc.config     import config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CRNN_CTC().to(device)
ckpt = torch.load(f"{DRIVE_PATH}/checkpoints_ctc/best_ctc.pt", map_location=device)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model.eval()

bank = build_multidigit_bank(config['data_path'])
clean_aug = get_digit_aug_pipeline(augment=False, config=config)

for L in [3, 7, 12, 20, 30]:
    # Force the target length by patching the config temporarily.
    # `make_sequence` will generate its own random digits of length L
    # and return them as `true_digits`.
    saved_min, saved_max = config['min_seq_len'], config['max_seq_len']
    config['min_seq_len'], config['max_seq_len'] = L, L
    img, true_digits = make_sequence(bank, clean_aug, config, augment=False, epoch=1)
    config['min_seq_len'], config['max_seq_len'] = saved_min, saved_max

    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))   # [1, T, 11]
    pred = greedy_decode(logits)[0]
    match = "OK" if pred == true_digits else "X"
    print(f"L={L:>2}  true={''.join(map(str, true_digits))}  pred={''.join(map(str, pred))}  {match}")

## 9. Inference on a Custom Image

Upload any digit-sequence image and run the CTC model end-to-end.

In [ ]:
from google.colab import files
uploaded = files.upload()
if uploaded:
    img_path = list(uploaded.keys())[0]
    !python -m src.ctc.inference \
        --image "{img_path}" \
        --checkpoint "{DRIVE_PATH}/checkpoints_ctc/best_ctc.pt" \
        --visualize